In [1]:
# ============================================================
# Q1 - Finding solutions of linear systems
#
# From-scratch REF, RREF, pivot detection,
# particular + null-space
#
# NO built-in rref / matrix_rank / solve are used anywhere.
# numpy is used ONLY to build/hold arrays and to sample randoms.
# ============================================================

import numpy as np


# ---- Reproducibility: fix the seed so printed numbers match the code ----
np.random.seed(42)

# Numerical tolerance:
# values with magnitude below this are treated as 0
TOL = 1e-9


# ------------------------------------------------------------
# Q1(1): Build the augmented matrix and reduce it to REF
# ------------------------------------------------------------

def to_ref(M):
    """
    Forward Gaussian elimination -> Row Echelon Form.

    Uses partial pivoting (largest-magnitude pivot), which also
    gives us an explicit, safe way to handle division-by-zero.
    """

    M = M.astype(float).copy()

    rows, cols = M.shape

    # Index of the next row to place a pivot in
    pivot_row = 0

    # Column indices that end up holding a pivot
    pivot_cols = []

    # Scan every column from left -> right
    for col in range(cols):

        # No rows left to place pivots in
        if pivot_row >= rows:
            break

        # Partial pivoting:
        # Pick the row at/below pivot_row with the largest |value|
        candidate = (
            np.argmax(np.abs(M[pivot_row:, col]))
            + pivot_row
        )

        # ----------------------------------------------------
        # DIVISION-BY-ZERO HANDLING
        # ----------------------------------------------------
        #
        # If even the largest candidate in this column is
        # approximately zero, the whole column is zero below
        # pivot_row.
        #
        # Therefore, there is NO valid pivot here.
        # We skip the column instead of dividing by 0.
        # ----------------------------------------------------

        if abs(M[candidate, col]) < TOL:
            continue

        # Swap the candidate row up into the pivot position
        M[[pivot_row, candidate]] = M[[candidate, pivot_row]]

        # Eliminate every entry BELOW the pivot
        # to create the staircase of zeros
        for r in range(pivot_row + 1, rows):

            # Safe because pivot is >= TOL
            factor = M[r, col] / M[pivot_row, col]

            M[r, :] = M[r, :] - factor * M[pivot_row, :]

        # Record this as a pivot column
        pivot_cols.append(col)

        # Next pivot goes in the next row down
        pivot_row += 1

    return M, pivot_cols


# ------------------------------------------------------------
# Q1(1): Continue from REF to RREF
# ------------------------------------------------------------

def to_rref(M):
    """
    Reduce to Reduced Row Echelon Form.

    Pivots are normalised to 1, and every entry ABOVE
    each pivot is cleared to 0.
    """

    # First get REF
    R, pivot_cols = to_ref(M)

    # Go from bottom pivot -> top pivot
    for i in reversed(range(len(pivot_cols))):

        col = pivot_cols[i]

        # After REF, pivot i is in row i
        row = i

        # Scale pivot entry to exactly 1
        R[row, :] = R[row, :] / R[row, col]

        # Clear entries ABOVE the pivot
        for r in range(row):
            R[r, :] = R[r, :] - R[r, col] * R[row, :]

    return R, pivot_cols


# ------------------------------------------------------------
# Q1(2): Pivot/non-pivot columns,
#         particular solution,
#         Ax = 0 solutions
# ------------------------------------------------------------

def analyse_system(A, b):
    """
    Given A (m x n) and b (m x 1), return:

        REF
        RREF
        pivot columns
        free columns
        particular solution
        basis for the null space
        consistency
    """

    m, n = A.shape

    # Construct augmented matrix [A | b]
    aug = np.hstack([
        A,
        b.reshape(-1, 1)
    ])

    # REF of augmented matrix
    REF, _ = to_ref(aug)

    # RREF of augmented matrix
    RREF, piv_all = to_rref(aug)

    # --------------------------------------------------------
    # Consistency check
    # --------------------------------------------------------
    #
    # A pivot in the LAST (constant) column means:
    #
    #       0 = nonzero
    #
    # Therefore, the system is inconsistent.
    # --------------------------------------------------------

    consistent = not any(c == n for c in piv_all)

    # Pivot columns belonging to A
    pivot_cols = [
        c for c in piv_all
        if c < n
    ]

    # Non-pivot columns are free-variable columns
    free_cols = [
        c for c in range(n)
        if c not in pivot_cols
    ]

    # --------------------------------------------------------
    # Particular solution
    # --------------------------------------------------------
    #
    # Set every free variable to 0.
    #
    # Then each pivot variable equals the constant on the
    # right-hand side of its pivot row.
    # --------------------------------------------------------

    x_p = np.zeros(n)

    for i, c in enumerate(pivot_cols):
        x_p[c] = RREF[i, n]

    # --------------------------------------------------------
    # Null space
    # --------------------------------------------------------
    #
    # All solutions of:
    #
    #       A x = 0
    #
    # There is one basis vector per free column.
    # --------------------------------------------------------

    null_basis = []

    for f in free_cols:

        # Start with all zeros
        v = np.zeros(n)

        # Set this free variable to 1
        v[f] = 1.0

        # Pivot variable:
        #
        # x_pivot = -(coefficient of free variable)
        #
        for i, c in enumerate(pivot_cols):
            v[c] = -RREF[i, f]

        null_basis.append(v)

    return (
        REF,
        RREF,
        pivot_cols,
        free_cols,
        x_p,
        null_basis,
        consistent
    )


# ------------------------------------------------------------
# Q1(3): Random 5 x 7 system
#         Show everything and verify
# ------------------------------------------------------------

np.set_printoptions(
    precision=8,
    suppress=True,
    linewidth=140
)


# Random FLOAT matrix and vector
# Standard-normal floats, NOT integers

A = np.random.randn(5, 7)

# 5 x 1 random floats
b = np.random.randn(5)


# ------------------------------------------------------------
# Display A and b
# ------------------------------------------------------------

print("A =\n", A)

print("\nb =\n", b)


# ------------------------------------------------------------
# Analyse the system
# ------------------------------------------------------------

(
    REF,
    RREF,
    pivot_cols,
    free_cols,
    x_p,
    null_basis,
    consistent
) = analyse_system(A, b)


# ------------------------------------------------------------
# Display augmented matrix
# ------------------------------------------------------------

print(
    "\nAugmented matrix [A | b] =\n",
    np.hstack([
        A,
        b.reshape(-1, 1)
    ])
)


# ------------------------------------------------------------
# Display REF
# ------------------------------------------------------------

print("\nREF of [A | b] =\n", REF)


# ------------------------------------------------------------
# Display RREF
# ------------------------------------------------------------

print("\nRREF of [A | b] =\n", RREF)


# ------------------------------------------------------------
# Display consistency
# ------------------------------------------------------------

print("\nConsistent system? ", consistent)

print("Pivot columns    :", pivot_cols)

print("Non-pivot columns:", free_cols)


# ------------------------------------------------------------
# Particular solution
# ------------------------------------------------------------

print("\nParticular solution x_p =\n", x_p)

print(
    "Check  A @ x_p =",
    A @ x_p,
    " (should equal b)"
)


# ------------------------------------------------------------
# Null-space basis
# ------------------------------------------------------------

print("\nNull-space basis (solutions to A x = 0):")

for j, v in enumerate(null_basis, 1):

    print(
        f"  n{j} =",
        v,
        "  -> A @ n{} =".format(j),
        A @ v,
        "(should be ~0)"
    )


# ------------------------------------------------------------
# General solution
#
# x = x_p + c1*n1 + c2*n2 + ...
# ------------------------------------------------------------

# Arbitrary free-variable values
coeffs = np.random.randn(len(null_basis))


# Construct general solution
x_general = (
    x_p
    + sum(
        c * v
        for c, v in zip(coeffs, null_basis)
    )
)


# ------------------------------------------------------------
# Verify general solution
# ------------------------------------------------------------

print(
    "\nRandom coefficients (c1, c2, ...):",
    coeffs
)

print(
    "General solution x =\n",
    x_general
)

print(
    "Verify  A @ x_general =",
    A @ x_general
)

print(
    "Target  b            =",
    b
)

print(
    "Max abs error        =",
    np.max(
        np.abs(
            A @ x_general - b
        )
    )
)

A =
 [[ 0.49671415 -0.1382643   0.64768854  1.52302986 -0.23415337 -0.23413696  1.57921282]
 [ 0.76743473 -0.46947439  0.54256004 -0.46341769 -0.46572975  0.24196227 -1.91328024]
 [-1.72491783 -0.56228753 -1.01283112  0.31424733 -0.90802408 -1.4123037   1.46564877]
 [-0.2257763   0.0675282  -1.42474819 -0.54438272  0.11092259 -1.15099358  0.37569802]
 [-0.60063869 -0.29169375 -0.60170661  1.85227818 -0.01349722 -1.05771093  0.82254491]]

b =
 [-1.22084365  0.2088636  -1.95967012 -1.32818605  0.19686124]

Augmented matrix [A | b] =
 [[ 0.49671415 -0.1382643   0.64768854  1.52302986 -0.23415337 -0.23413696  1.57921282 -1.22084365]
 [ 0.76743473 -0.46947439  0.54256004 -0.46341769 -0.46572975  0.24196227 -1.91328024  0.2088636 ]
 [-1.72491783 -0.56228753 -1.01283112  0.31424733 -0.90802408 -1.4123037   1.46564877 -1.95967012]
 [-0.2257763   0.0675282  -1.42474819 -0.54438272  0.11092259 -1.15099358  0.37569802 -1.32818605]
 [-0.60063869 -0.29169375 -0.60170661  1.85227818 -0.01349722 -1.0

In [2]:
# ============================================================
# Q2 - Dataset, Rank, Covariance, Power Method
#
# Power method + rank are hand-written.
# np.linalg.eigh is used ONLY in part (d), which explicitly
# asks for the library solver.
# ============================================================

import numpy as np


# ------------------------------------------------------------
# Reproducibility and numerical settings
# ------------------------------------------------------------

# Fixed seed so the dataset and every number below
# are reproducible.
np.random.seed(42)

np.set_printoptions(
    precision=8,
    suppress=True,
    linewidth=140
)

# Tolerance for treating a value as zero
# (used for rank counting)
TOL = 1e-8


# ------------------------------------------------------------
# Manual Euclidean norm
#
# No np.linalg.norm is used.
#
# ||x|| = sqrt(x^T x)
# ------------------------------------------------------------

def vnorm(x):
    """
    Compute the Euclidean norm of vector x manually.

    ||x|| = sqrt(sum of squares)
    """
    return (x @ x) ** 0.5


# ------------------------------------------------------------
# Q2(1): Generate X in R^(500 x 6)
#
# f1 ... f4 ~ standard normal
#
# f5 = 2*f1 + 3*f2
# f6 = f3 - 2*f4
#
# Therefore, f5 and f6 are linearly dependent on
# the first four features.
# ------------------------------------------------------------

n = 500  # Number of data points (rows)


# First four independent random features
f1 = np.random.randn(n)
f2 = np.random.randn(n)
f3 = np.random.randn(n)
f4 = np.random.randn(n)


# Engineered linearly-dependent features
f5 = 2 * f1 + 3 * f2
f6 = f3 - 2 * f4


# Assemble the 500 x 6 dataset
X = np.column_stack([
    f1,
    f2,
    f3,
    f4,
    f5,
    f6
])


print("Q2(1)  Dataset X")
print("Shape of X:", X.shape)
print("First 10 rows of X:\n", X[:10])


# ------------------------------------------------------------
# Q2(2): Rank of X WITHOUT np.linalg.matrix_rank
#
# Reduce X to REF using Gaussian elimination and count
# the number of pivots.
# ------------------------------------------------------------

def rank_via_elimination(M):
    """
    Compute the rank of M using Gaussian elimination.

    Rank = number of pivots obtained during elimination.
    """

    # Work on a floating-point copy
    M = M.astype(float).copy()

    rows, cols = M.shape

    pivot_row = 0
    rank = 0

    # Sweep through each column
    for col in range(cols):

        # No rows left for additional pivots
        if pivot_row >= rows:
            break

        # Partial pivoting:
        # choose the row with the largest absolute value
        candidate = (
            np.argmax(
                np.abs(M[pivot_row:, col])
            )
            + pivot_row
        )

        # If the largest value is approximately zero,
        # there is no pivot in this column.
        if abs(M[candidate, col]) < TOL:
            continue

        # Swap the candidate row into the pivot position
        M[[pivot_row, candidate]] = (
            M[[candidate, pivot_row]]
        )

        # Eliminate entries below the pivot
        for r in range(pivot_row + 1, rows):

            M[r, :] -= (
                M[r, col] / M[pivot_row, col]
            ) * M[pivot_row, :]

        # One additional pivot found
        rank += 1

        # Move to the next pivot row
        pivot_row += 1

    return rank


print("\nQ2(2)  Rank of X")

print(
    "rank(X) =",
    rank_via_elimination(X),
    " (expected 4: f5, f6 are linear combinations "
    "of f1..f4)"
)


# ------------------------------------------------------------
# Q2(3a): Covariance matrix
#
# C = (1/n) X^T X
#
# Since the features have approximately zero mean because
# they were generated from standard normal variables,
# this is the covariance matrix specified in the question.
# ------------------------------------------------------------

C = (X.T @ X) / n


print("\nQ2(3a)  Covariance matrix C = (1/n) X^T X")

print("Shape of C:", C.shape)

print("C =\n", C)


# ------------------------------------------------------------
# Q2(3b): Power Method
#
# Find the dominant (largest) eigenpair.
# ------------------------------------------------------------

def power_method(
    M,
    max_iter=100000,
    tol=1e-12,
    seed=0
):
    """
    Return:

        dominant eigenvalue
        unit eigenvector
        iteration count
        eigenvalue history

    The eigenvalue is estimated using the Rayleigh quotient:

        lambda = v^T M v
    """

    # Random starting vector
    rng = np.random.default_rng(seed)

    v = rng.standard_normal(M.shape[0])

    # Normalise the initial vector
    v = v / vnorm(v)

    # Previous eigenvalue estimate
    lam_old = 0.0

    # Store eigenvalue estimates from every iteration
    history = []

    # Power iterations
    for k in range(1, max_iter + 1):

        # Apply matrix to current vector
        w = M @ v

        # Renormalise
        v = w / vnorm(w)

        # Rayleigh quotient
        # lambda = v^T M v
        lam = v @ (M @ v)

        # Save eigenvalue estimate
        history.append(lam)

        # Check convergence
        if abs(lam - lam_old) < tol:
            return (
                lam,
                v,
                k,
                history
            )

        # Update previous estimate
        lam_old = lam

    # Maximum number of iterations reached
    return (
        lam,
        v,
        max_iter,
        history
    )


# Run power method on covariance matrix C
lam1, v1, it1, _ = power_method(
    C,
    seed=1
)


print("\nQ2(3b)  Power Method - dominant eigenpair")

print("lambda_1 =", lam1)

print("v_1      =", v1)

print(
    "iterations to internal convergence:",
    it1
)


# ------------------------------------------------------------
# Q2(3c): Deflation
#
# Extract the next eigenpairs by removing directions
# corresponding to eigenvectors already found.
#
# Formula specified in Q2(c):
#
#     M = C - (sum_j v_j v_j^T) C
#
# For each previously found eigenvector v_j:
#
#     M = M - v_j v_j^T M
# ------------------------------------------------------------

def power_method_deflation(C, k, seed=1):
    """
    Extract k eigenpairs using the Power Method with deflation.

    Before each new Power Method run, previously found
    eigenvector directions are removed.
    """

    eigvals = []
    eigvecs = []

    # Extract k eigenpairs
    for i in range(k):

        # Start with the original covariance matrix
        M = C.copy()

        # Remove previously found eigenvector directions
        for vj in eigvecs:

            # Deflation:
            #
            # M = M - v_j v_j^T M
            #
            M = (
                M
                - np.outer(vj, vj) @ M
            )

        # Run Power Method on deflated matrix
        lam, v, _, _ = power_method(
            M,
            seed=seed + i
        )

        # Store eigenpair
        eigvals.append(lam)
        eigvecs.append(v)

    return (
        np.array(eigvals),
        np.array(eigvecs)
    )


# rank(X) = 4
# Therefore, there are only 4 non-zero eigenvalues.
k = 4


pm_vals, pm_vecs = power_method_deflation(
    C,
    k
)


print(
    "\nQ2(3c)  Deflation - top",
    k,
    "eigenpairs (rank of C is 4)"
)


for i in range(k):

    print(
        f"lambda_{i + 1} = {pm_vals[i]: .8f}"
        f"   v_{i + 1} = {pm_vecs[i]}"
    )


# ------------------------------------------------------------
# Q2(3d): Library eigensolver
#
# np.linalg.eigh is ALLOWED here because part (d)
# explicitly asks for the library solver.
#
# eigh is appropriate because C is symmetric.
# ------------------------------------------------------------

w, V = np.linalg.eigh(C)


# Sort eigenvalues from largest -> smallest
order = np.argsort(w)[::-1]

w = w[order]
V = V[:, order]


print(
    "\nQ2(3d)  Library eigensolver np.linalg.eigh"
)

print(
    "All eigenvalues (desc):",
    w
)


for i in range(k):

    print(
        f"lambda_{i + 1} (lib) = {w[i]: .8f}"
        f"   v_{i + 1} (lib) = {V[:, i]}"
    )


# ------------------------------------------------------------
# Compare Power Method with library eigensolver
#
# Eigenvectors are unique only up to sign:
#
#     v and -v
#
# represent the same eigenvector direction.
#
# Therefore, check the sign using the dot product.
# ------------------------------------------------------------

print(
    "\nComparison (power method vs library) "
    "for the 4 non-zero eigenvalues:"
)


for i in range(k):

    # Determine whether the Power Method vector and
    # library vector point in the same or opposite direction.
    same_dir = np.sign(
        pm_vecs[i] @ V[:, i]
    )

    # Difference after correcting for possible sign flip
    vec_diff = vnorm(
        pm_vecs[i]
        - same_dir * V[:, i]
    )

    # Eigenvalue difference
    eig_diff = abs(
        pm_vals[i] - w[i]
    )

    print(
        f"  lambda_{i + 1}: "
        f"PM={pm_vals[i]: .8f}  "
        f"LIB={w[i]: .8f}  "
        f"|diff|={eig_diff:.2e}"
        f"  vec |diff|={vec_diff:.2e}"
    )


# ------------------------------------------------------------
# Q2(3e): Number of iterations required to reach
# accuracy 1e-7 compared with library eigenvalues
# ------------------------------------------------------------

def iters_to_accuracy(
    C,
    eigvecs_found,
    true_lambda,
    target=1e-7,
    seed=1,
    idx=0
):
    """
    Count the number of Power Method iterations required
    to achieve:

        |lambda_k - true_lambda| < target

    Previously found eigenvectors are deflated first.
    """

    # Start with the original covariance matrix
    M = C.copy()

    # Deflate previously found eigenvector directions
    for vj in eigvecs_found:

        M = (
            M
            - np.outer(vj, vj) @ M
        )

    # Run Power Method
    _, _, _, hist = power_method(
        M,
        seed=seed + idx
    )

    # Search through the eigenvalue history
    for k_iter, lam in enumerate(hist, 1):

        if abs(lam - true_lambda) < target:
            return k_iter

    # If target was not reached, return the
    # total number of iterations performed.
    return len(hist)


print(
    "\nQ2(3e)  Iterations to reach accuracy 1e-7 "
    "(true values from part d)"
)


# Previously found Power Method eigenvectors
found = []


for i in range(k):

    # Number of iterations needed
    it = iters_to_accuracy(
        C,
        found,
        w[i],
        idx=i
    )

    # Eigenvalue gap ratio
    if i + 1 < len(w):
        gap_ratio = abs(
            w[i + 1] / w[i]
        )
    else:
        gap_ratio = 0

    print(
        f"  lambda_{i + 1}: "
        f"{it} iterations   "
        f"(gap ratio "
        f"|lam_{i + 2}/lam_{i + 1}| "
        f"= {gap_ratio:.4f})"
    )

    # Add current eigenvector to the list
    # for deflation in the next iteration.
    found.append(pm_vecs[i])

Q2(1)  Dataset X
Shape of X: (500, 6)
First 10 rows of X:
 [[ 0.49671415  0.92617755  1.39935544  0.77836108  3.77196095 -0.15736672]
 [-0.1382643   1.90941664  0.92463368 -0.55118572  5.45172132  2.02700512]
 [ 0.64768854 -1.39856757  0.05963037 -0.81819888 -2.90032565  1.69602814]
 [ 1.52302986  0.56296924 -0.64693678 -0.00337446  4.73496742 -0.64018786]
 [-0.23415337 -0.65064257  0.69822331 -0.17018462 -2.42023446  1.03859256]
 [-0.23413696 -0.48712538  0.39348539 -0.45322805 -1.92965007  1.29994148]
 [ 1.57921282 -0.59239392  0.89519322  0.69638745  1.38124386 -0.49758167]
 [ 0.76743473 -0.86399077  0.6351718   0.95530521 -1.05710285 -1.27543862]
 [-0.46947439  0.04852163  1.04955272  0.08840689 -0.79338389  0.87273894]
 [ 0.54256004 -0.83095012 -0.53523521  1.47753008 -1.40773026 -3.49029537]]

Q2(2)  Rank of X
rank(X) = 4  (expected 4: f5, f6 are linear combinations of f1..f4)

Q2(3a)  Covariance matrix C = (1/n) X^T X
Shape of C: (6, 6)
C =
 [[ 0.96097898 -0.07225555 -0.05643179